In [1]:
!pip install -q -U langchain-google-genai langchain langchain-core

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 1.5.1 which is incompatible.
langchain-community 0.3.24 requires langchain<1.0.0,>=0.3.25, but you have langchain 1.3.14 which is incompatible.
langchain-community 0.3.24 requires langchain-core<1.0.0,>=0.3.59, but you have langchain-core 1.5.1 which is incompatible.


In [2]:
!pip uninstall -y langchain langchain-core langchain-community langchain-google-genai

Found existing installation: langchain 1.3.14
Uninstalling langchain-1.3.14:
  Successfully uninstalled langchain-1.3.14
Found existing installation: langchain-core 1.5.1
Uninstalling langchain-core-1.5.1:
  Successfully uninstalled langchain-core-1.5.1
Found existing installation: langchain-community 0.3.24
Uninstalling langchain-community-0.3.24:
  Successfully uninstalled langchain-community-0.3.24
Found existing installation: langchain-google-genai 4.3.1
Uninstalling langchain-google-genai-4.3.1:
  Successfully uninstalled langchain-google-genai-4.3.1


In [3]:
!pip install \
langchain==0.3.25 \
langchain-core==0.3.59 \
langchain-community==0.3.24 \
langchain-google-genai

  Using cached langchain-0.3.25-py3-none-any.whl.metadata (7.8 kB)
  Using cached langchain_core-0.3.59-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_community-0.3.24-py3-none-any.whl.metadata (2.5 kB)
  Using cached langchain_google_genai-4.3.1-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_google_genai-4.3.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.2.7-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.2.6-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.2.5-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.2.4-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.2.3-py3-none-any.whl.metadata (2.7 kB)
  Using cached google_genai-1.75.0-py3-none-any.whl.metadata (52 kB)
  U

In [4]:
# Business Rules Thresholds (Constants)
MAX_AUTO_APPROVAL_AMOUNT = 10000
FRAUD_SCORE_THRESHOLD = 70
MAX_ALLOWED_PREVIOUS_CLAIMS = 5


def process_claim_reactive(claim_data: dict) -> dict:
    """
    Reactive Rule-Based Claims Triage Agent.

    This agent uses fixed business rules to classify insurance claims.
    No LLM, reasoning, memory, or tool usage is involved.
    The first matching rule determines the final decision.
    """
    claim_amount = claim_data.get("claim_amount", 0)
    policy_status = claim_data.get("policy_status", "").upper()
    fraud_score = claim_data.get("fraud_score", 0)
    previous_claims = claim_data.get("previous_claims", 0)
    has_missing_docs = claim_data.get("has_missing_docs", False)

    # Rule 1: Policy Inactive -> REJECT
    if policy_status != "ACTIVE":
        return {
            "decision": "REJECT",
            "reason": "Policy is not active."
        }

    # Rule 2: Missing Documents -> ESCALATE
    if has_missing_docs:
        return {
            "decision": "ESCALATE",
            "reason": "Required documentation is missing."
        }

    # Rule 3: High Fraud Score -> ESCALATE
    if fraud_score >= FRAUD_SCORE_THRESHOLD:
        return {
            "decision": "ESCALATE",
            "reason": "High fraud score detected."
        }

    # Rule 4: Too Many Previous Claims -> ESCALATE
    if previous_claims >= MAX_ALLOWED_PREVIOUS_CLAIMS:
        return {
            "decision": "ESCALATE",
            "reason": "Customer has many previous claims."
        }

    # Rule 5: High Claim Amount -> ESCALATE
    if claim_amount > MAX_AUTO_APPROVAL_AMOUNT:
        return {
            "decision": "ESCALATE",
            "reason": "Claim amount exceeds automatic approval limit."
        }

    # Default Rule: If all criteria pass -> APPROVE
    return {
        "decision": "APPROVE",
        "reason": "All basic criteria met successfully."
    }


# Execution & Testing


claims = [
    {
        "claim_id": "CLM-101",
        "claim_amount": 12000,
        "policy_status": "ACTIVE",
        "fraud_score": 45,
        "previous_claims": 8,
        "has_missing_docs": False
    },
    {
        "claim_id": "CLM-102",
        "claim_amount": 5000,
        "policy_status": "ACTIVE",
        "fraud_score": 20,
        "previous_claims": 1,
        "has_missing_docs": False
    },
    {
        "claim_id": "CLM-103",
        "claim_amount": 3000,
        "policy_status": "EXPIRED",
        "fraud_score": 10,
        "previous_claims": 0,
        "has_missing_docs": False
    }
]

for claim in claims:
    result = process_claim_reactive(claim)

    print("=" * 60)
    print(f"Claim ID        : {claim['claim_id']}")
    print(f"Amount          : ${claim['claim_amount']}")
    print(f"Policy Status   : {claim['policy_status']}")
    print(f"Fraud Score     : {claim['fraud_score']}")
    print(f"Previous Claims : {claim['previous_claims']}")
    print(f"Missing Docs    : {claim['has_missing_docs']}")
    print("-" * 60)
    print(f"Decision        : {result['decision']}")
    print(f"Reason          : {result['reason']}")

Claim ID        : CLM-101
Amount          : $12000
Policy Status   : ACTIVE
Fraud Score     : 45
Previous Claims : 8
Missing Docs    : False
------------------------------------------------------------
Decision        : ESCALATE
Reason          : Customer has many previous claims.
Claim ID        : CLM-102
Amount          : $5000
Policy Status   : ACTIVE
Fraud Score     : 20
Previous Claims : 1
Missing Docs    : False
------------------------------------------------------------
Decision        : APPROVE
Reason          : All basic criteria met successfully.
Claim ID        : CLM-103
Amount          : $3000
Policy Status   : EXPIRED
Fraud Score     : 10
Previous Claims : 0
Missing Docs    : False
------------------------------------------------------------
Decision        : REJECT
Reason          : Policy is not active.


In [11]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_react_agent, AgentExecutor


llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=os.environ.get("GOOGLE_API_KEY"),
    temperature=0.2
)


@tool
def get_claim_information(claim_id: str) -> dict:
    """Retrieves basic claim details including claim amount, missing documents status, and description."""
    mock_db = {
        "CLM-101": {"claim_amount": 12000, "has_missing_docs": False, "description": "Vehicle accident repair request."},
        "CLM-102": {"claim_amount": 5000, "has_missing_docs": False, "description": "Windshield damage replacement."},
        "CLM-103": {"claim_amount": 3000, "has_missing_docs": False, "description": "Minor bumper repair."}
    }
    return mock_db.get(claim_id, {"error": "Claim ID not found"})


@tool
def check_policy_status(claim_id: str) -> dict:
    """Checks the status of the insurance policy for a given claim ID."""
    mock_db = {
        "CLM-101": "ACTIVE",
        "CLM-102": "ACTIVE",
        "CLM-103": "EXPIRED"
    }
    status = mock_db.get(claim_id, "UNKNOWN")
    return {"claim_id": claim_id, "policy_status": status}


@tool
def check_fraud_and_history(claim_id: str) -> dict:
    """Retrieves fraud score and previous claim history for a given claim ID."""
    mock_db = {
        "CLM-101": {"fraud_score": 45, "previous_claims": 8},
        "CLM-102": {"fraud_score": 20, "previous_claims": 1},
        "CLM-103": {"fraud_score": 10, "previous_claims": 0}
    }
    data = mock_db.get(claim_id, {"fraud_score": 0, "previous_claims": 0})
    return {
        "claim_id": claim_id,
        "fraud_score": data["fraud_score"],
        "previous_claims": data["previous_claims"]
    }


@tool
def check_claim_amount(amount: int) -> dict:
    """Evaluates the claim amount against financial approval thresholds."""
    is_high_value = amount > 10000
    return {
        "amount": amount,
        "category": "High-value claim" if is_high_value else "Standard-value claim",
        "requires_manual_review": is_high_value
    }


tools = [
    get_claim_information,
    check_policy_status,
    check_fraud_and_history,
    check_claim_amount
]

# ---------------------------------------------------------
# 3. Unconstrained ReAct Agent
# ---------------------------------------------------------

prompt_template = """Answer the following question as best you can.

You have access to the following tools:

{tools}

Use the following format:

Question: the input question

Thought: think about what to do

Action: one of [{tool_names}]

Action Input: the input to the action

Observation: the result of the action

...(repeat Thought/Action/Observation as needed)...

Thought: I now know the final answer

Final Answer: your final answer

Question: {input}

Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(prompt_template)

agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    return_intermediate_steps=True
)

# 4.(Execution & Testing)

claims_to_test = ["CLM-101", "CLM-102", "CLM-103"]

for claim_id in claims_to_test:
    query = f"""
    You are an unconstrained ReAct insurance claims triage agent.

    Reason step-by-step.
    Choose tools only when necessary.
    You may call the same tool multiple times before reaching a decision.

    Triage the insurance claim for Claim ID: {claim_id}.
    You must fetch all necessary details using the available tools before making a decision.

    Finally, decide whether to APPROVE, REJECT, or ESCALATE, and provide a clear explanation for your decision.
    """

    print("=" * 70)
    print(f"Processing Claim ID: {claim_id}")
    print("=" * 70)

    response = agent_executor.invoke({"input": query})

    print("\n" + "-" * 30 + " FULL RESPONSE OBJECT " + "-" * 30)
    print(response)
    print("\n")

Processing Claim ID: CLM-101


> Entering new AgentExecutor chain...


ChatGoogleGenerativeAIError: Invalid argument provided to Gemini: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]